In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/pambifree/02-data-audit/__huggingface_repos__.json
/kaggle/input/notebooks/pambifree/02-data-audit/__notebook__.ipynb
/kaggle/input/notebooks/pambifree/02-data-audit/__output__.json
/kaggle/input/notebooks/pambifree/02-data-audit/__results__.html
/kaggle/input/notebooks/pambifree/02-data-audit/custom.css
/kaggle/input/notebooks/pambifree/02-data-audit/gold_test_v1.csv
/kaggle/input/notebooks/pambifree/02-data-audit/public_mix_v1/test.csv
/kaggle/input/notebooks/pambifree/02-data-audit/public_mix_v1/train.csv
/kaggle/input/notebooks/pambifree/02-data-audit/public_mix_v1/val.csv


In [2]:
!find /kaggle/input -name "*.csv" | head -20

/kaggle/input/notebooks/pambifree/02-data-audit/gold_test_v1.csv
/kaggle/input/notebooks/pambifree/02-data-audit/public_mix_v1/test.csv
/kaggle/input/notebooks/pambifree/02-data-audit/public_mix_v1/train.csv
/kaggle/input/notebooks/pambifree/02-data-audit/public_mix_v1/val.csv


In [3]:
import pandas as pd, numpy as np

PUBLIC_DIR = "/kaggle/input/notebooks/pambifree/02-data-audit/public_mix_v1"   # replace with your path
GOLD_PATH = "/kaggle/input/notebooks/pambifree/02-data-audit/gold_test_v1.csv" # replace with your path

train = pd.read_csv(f"{PUBLIC_DIR}/train.csv")
val = pd.read_csv(f"{PUBLIC_DIR}/val.csv")
test = pd.read_csv(f"{PUBLIC_DIR}/test.csv")
gold = pd.read_csv(GOLD_PATH)

print(train.shape, val.shape, test.shape, gold.shape)
print(gold["label"].value_counts().to_dict())


(7154, 5) (1485, 5) (2066, 5) (326, 7)
{1: 183, 0: 143}


In [4]:
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from sklearn.model_selection import StratifiedGroupKFold

SEED = 42

def cap_groups(df, cap=5, seed=SEED):
    """Keep at most `cap` rows from each template family."""
    return df.sample(frac=1, random_state=seed).groupby("group").head(cap)

def scores(y_true, y_pred):
    p, r, f, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", pos_label=1, zero_division=0)
    return {"precision": p, "recall": r, "f1": f, "accuracy": accuracy_score(y_true, y_pred)}

def make_models():
    return {
        "Majority class": Pipeline([("vec", CountVectorizer()),
                                    ("clf", DummyClassifier(strategy="most_frequent"))]),
        "Naive Bayes (word counts)": Pipeline([("vec", CountVectorizer()),
                                               ("clf", MultinomialNB())]),
        "TF-IDF words + LogReg": Pipeline([
            ("vec", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
            ("clf", LogisticRegression(max_iter=2000))]),
        "TF-IDF chars + LogReg": Pipeline([
            ("vec", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True)),
            ("clf", LogisticRegression(max_iter=2000))]),
    }

In [5]:
trainval = pd.concat([train, val], ignore_index=True)
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)

rows = []
for fold, (tr_idx, te_idx) in enumerate(sgkf.split(trainval, trainval["is_scam"], groups=trainval["group"])):
    tr, te = trainval.iloc[tr_idx], trainval.iloc[te_idx]
    te_c = cap_groups(te)
    for name, model in make_models().items():
        model.fit(tr["text"], tr["is_scam"])
        rows.append({"model": name, "fold": fold,
                     **scores(te_c["is_scam"], model.predict(te_c["text"]))})

cv = pd.DataFrame(rows)
cv_summary = cv.groupby("model")[["precision", "recall", "f1", "accuracy"]].agg(["mean", "std"]).round(3)
cv_summary

precision        recall            f1         \
                               mean    std   mean    std   mean    std   
model                                                                    
Majority class                0.414  0.232  0.800  0.447  0.546  0.305   
Naive Bayes (word counts)     0.923  0.007  0.963  0.006  0.943  0.005   
TF-IDF chars + LogReg         0.959  0.005  0.969  0.010  0.964  0.006   
TF-IDF words + LogReg         0.949  0.005  0.962  0.007  0.956  0.004   

                          accuracy         
                              mean    std  
model                                      
Majority class               0.510  0.020  
Naive Bayes (word counts)    0.939  0.005  
TF-IDF chars + LogReg        0.963  0.007  
TF-IDF words + LogReg        0.954  0.005

In [6]:
test_c = cap_groups(test)
eval_sets = [("public test (capped)", test_c, "is_scam"),
             ("public test (all rows)", test, "is_scam"),
             ("gold v1", gold, "label")]

fitted, final_rows = {}, []
for name, model in make_models().items():
    model.fit(trainval["text"], trainval["is_scam"])
    fitted[name] = model
    for set_name, df_, ycol in eval_sets:
        final_rows.append({"eval set": set_name, "model": name,
                           **scores(df_[ycol], model.predict(df_["text"]))})

final = pd.DataFrame(final_rows).round(3).set_index(["eval set", "model"])
final

,,precision,recall,f1,accuracy
eval set,model,,,,
public test (capped),Majority class,0.528,1.000,0.691,0.528
public test (all rows),Majority class,0.399,1.000,0.571,0.399
gold v1,Majority class,0.561,1.000,0.719,0.561
public test (capped),Naive Bayes (word counts),0.911,0.963,0.936,0.931
public test (all rows),Naive Bayes (word counts),0.587,0.967,0.730,0.715
gold v1,Naive Bayes (word counts),0.624,0.978,0.762,0.656
public test (capped),TF-IDF words + LogReg,0.932,0.958,0.945,0.941
public test (all rows),TF-IDF words + LogReg,0.593,0.962,0.734,0.722
gold v1,TF-IDF words + LogReg,0.644,0.891,0.748,0.663


In [7]:
best = fitted["TF-IDF chars + LogReg"]
gold["pred"] = best.predict(gold["text"])
gold["correct"] = gold["pred"] == gold["label"]

print("By type (legit = correctly left alone; scam/promo = correctly caught):")
print(gold.groupby("type")["correct"].agg(["mean", "size"]).round(2), "\n")

print("By tag, worst first:")
print(gold.groupby("tag")["correct"].agg(["mean", "size"]).round(2).sort_values("mean").head(12), "\n")

print("By origin:")
print(gold.groupby("origin")["correct"].agg(["mean", "size"]).round(2), "\n")

print("Ten mistakes:")
for t, y in zip(gold[~gold["correct"]]["text"].head(10), gold[~gold["correct"]]["label"].head(10)):
    print(f"[true={y}]", t[:120])

By type (legit = correctly left alone; scam/promo = correctly caught):
       mean  size
type             
legit  0.51   143
promo  0.91    65
scam   0.81   118 

By tag, worst first:
                mean  size
tag                       
has_link        0.14    21
alarming        0.20    20
hinglish_alert  0.20    15
transactional   0.30    20
scam_warning    0.30    10
very_long       0.50     2
obfuscated      0.56    16
shorthand       0.67    24
promo_hin       0.76    25
other_scam      0.77    13
extortion_scam  0.80     5
hinglish_scam   0.84    25 

By origin:
        mean  size
origin            
human   0.76    17
llm     0.69   309 

Ten mistakes:
[true=1] Hi Papa/Mummy, this is my new number. Can you send me 18,000 urgently as I need to replace my phone as the old one fell 
[true=1] Are you free? I'm in a meeting, need you to buy Amazon gift cards for a client. Will reimburse.
[true=1] Mam, you son has been arrested and is in our custody. Send us the bail money or will be i

In [8]:
cv_summary.to_csv("/kaggle/working/baseline_cv_summary.csv")
final.to_csv("/kaggle/working/baseline_final_results.csv")